# AIワークショップ用 Colab接続テスト

**所要時間：約3分。文字の入力や設定の変更は不要です。**

## 手順1
1. 下にあるコードの左上の **[ ]**（マウスを重ねると **▶** に変わります）を1回押してください。
2. 「**警告：このノートブックは Google が作成したものではありません**」と出たら **「このまま実行」** を押してください。
3. 30秒〜1分ほどで結果が表示されます。結果の中の **「↓↓↓ ここから ↓↓↓」から「↑↑↑ ここまで ↑↑↑」まで** をマウスでなぞって選択し、コピー（Windowsは Ctrl＋C、Macは Command＋C）して、**ご案内メールへの返信に貼り付けてください**。
4. そのまま次の **手順2** に進んでください。

## うまくいかないとき
- ログインを求められたら、**ワークショップ当日に使う予定または同条件のGoogleアカウント**でログインしてください。
- 「接続中」のまま1分以上進まない、またはエラーが出て結果が表示されない場合は、**画面に表示された内容**をメールでお知らせください。それ自体が今回の確認結果です。


In [ ]:
# @title
# ======================================================
#  AI研修 接続テスト ― 何も書き換えずに左の ▶ を押してください
# ======================================================
# （下の2行だけ、講師側が事前に設定します）
FILE_URL = "https://raw.githubusercontent.com/mkuriyama/ai-strategy-toolkit/refs/heads/main/connection_test/ping.txt"  # 教材と同じ置き場の小さなファイル
GAS_URL  = "https://script.google.com/macros/s/AKfycbwzU2VFjDwd4RQlBFOABohqHvry3_W6hDT9mBZF50vZQNSXP1Dy54EJ0g7rvaJ9bLzwMw/exec"   # テスト用GASのWebアプリURL（…/exec）。空欄ならこの項目はスキップ

import time, math, datetime, requests

print("テストを開始します。1分ほどお待ちください…\n")
results = []

def check(no, label, fn):
    t0 = time.time()
    try:
        detail, ok = fn(), True
    except Exception as e:
        detail, ok = f"{type(e).__name__}: {str(e)[:90]}", False
    sec = time.time() - t0
    results.append((no, label, ok, detail, sec))
    print(f" {'✅' if ok else '❌'} {no}. {label}  ({sec:.1f}秒)")
    if not ok:
        print(f"      → {detail}")

def get(url):
    return requests.get(url, timeout=20, allow_redirects=True)

def code_so_far():
    return "-".join("OK" if r[2] else ("SKIP" if r[2] is None else "NG") for r in results)

# 1. 実行環境（このセルが動いた時点で接続できています）
check(1, "Colabの実行環境に接続", lambda: "ok")

# 2. 教材ファイルの取得
def f2():
    r = get(FILE_URL)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}")
    return f"{len(r.content)} bytes"
check(2, "教材ファイルの取得（GitHub）", f2)

# 3. AIサーバー（鍵なしで到達確認のみ。応答があれば到達OK）
def f3():
    r = get("https://generativelanguage.googleapis.com/v1beta/models")
    return f"HTTP {r.status_code}"
check(3, "AIサーバー（Gemini API）への到達", f3)

# 4. 地図サーバー（国土地理院の航空写真タイル：東京駅付近）
def f4():
    lat, lon, z = 35.68, 139.76, 15
    x = int((lon + 180) / 360 * 2**z)
    y = int((1 - math.log(math.tan(math.radians(lat)) + 1 / math.cos(math.radians(lat))) / math.pi) / 2 * 2**z)
    r = get(f"https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.jpg")
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}")
    return f"{len(r.content)} bytes"
check(4, "地図サーバー（国土地理院）への到達", f4)

# 5. 投稿先（Google Apps Script）― ここまでの結果を送信して記録してもらう
if GAS_URL:
    def f5():
        payload = {
            "kind": "connection_test",
            "judge": "NG" if any(r[2] is False for r in results) else "OK",
            "code": code_so_far(),
            "note": "notebook v04",
        }
        r = requests.post(GAS_URL, json=payload, timeout=30, allow_redirects=True)
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}")
        try:
            if r.json().get("ok"):
                return "記録されました"
        except Exception:
            pass
        return "応答あり（記録は未確認）"
    check(5, "投稿先（script.google.com）への送信", f5)
else:
    results.append((5, "投稿先（script.google.com）への送信", None, "未設定のためスキップ", 0.0))
    print(" ⏭  5. 投稿先（script.google.com）への送信  （未設定のためスキップ）")

# ---- まとめ（メールに貼り付けていただく部分） ----
ng   = [r for r in results if r[2] is False]
now  = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=9))).strftime("%Y-%m-%d %H:%M")
short = {1: "実行環境", 2: "教材ファイル", 3: "AIサーバー", 4: "地図サーバー", 5: "投稿先"}
print("\n\n↓↓↓ ここから ↓↓↓")
print(f"【接続テスト結果】{now}")
for no, label, ok, detail, sec in results:
    mark = "OK" if ok else ("—" if ok is None else "NG")
    if ok is False:
        kind, _, msg = detail.partition(": ")
        extra = f"（{msg[:30] if kind == 'RuntimeError' else kind}）"
    else:
        extra = ""
    print(f"{no}. {short.get(no, label)}：{mark}{extra}")
print("判定：すべてOK" if not ng else f"判定：{len(ng)}項目がNG")
print(f"確認コード：{code_so_far()}")
print("↑↑↑ ここまで ↑↑↑\n")
print("▶ 上の「ここから」〜「ここまで」をコピーしてメールに貼り付けたら、")
print("　 下の「手順2」に進んでください。")


## 手順2（30秒）結果一覧画面の確認

ワークショップ中に皆さんの結果を一覧表示する画面が、このPCから見られるかの確認です。
次のリンクを開いてください。

👉 [結果一覧画面を開く](https://script.google.com/macros/s/AKfycbwzU2VFjDwd4RQlBFOABohqHvry3_W6hDT9mBZF50vZQNSXP1Dy54EJ0g7rvaJ9bLzwMw/exec)

「**ギャラリー表示テスト：OK**」と表示されたら、画面内の **「↓↓↓ ここから ↓↓↓」から「↑↑↑ ここまで ↑↑↑」まで** を、手順1と同じように**同じメールに貼り付けてください**。

- 画面上部に「このアプリケーションは Google Apps Script のユーザーによって作成されたものです」と出ますが、Googleが自動で表示する案内ですので問題ありません。
- 表示されない（真っ白・エラー・ブロックの画面など）場合は、その旨をメールにお書きください。

---
以上で終了です。ご協力ありがとうございました。
